In [1]:
# NORTHSTAR -- Paid Features & Tier-Gated QA for Solace Browser
# DNA: tiers(Free/Starter/Pro) = pricing(correct) x cloud_twin(Pro+) x byok(settings) x evidence(retention) x auth_gate(upgrade)
# Probes settings.html, start.html, schedule.html for tier references and gating
import json, re, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "paid-features-qa"
NOTEBOOK_PATH = "notebooks/qa/33-paid-features-qa.ipynb"
MIN_SCORE = 70

BROWSER = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER / "web"

results = []

def record(probe_id, name, passed, detail="", tower_ref=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status,
                    "detail": detail, "tower_ref": tower_ref})
    print(f"  {status}: {name}" + (f" -- {detail}" if detail and not passed else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

# Pre-load files
settings_html = read_file(WEB / "settings.html")
start_html = read_file(WEB / "start.html")
schedule_html = read_file(WEB / "schedule.html")

print("Paid Features & Tier-Gated QA")
print(f"  settings.html: {len(settings_html)} bytes")
print(f"  start.html:    {len(start_html)} bytes")
print(f"  schedule.html: {len(schedule_html)} bytes")
assert BROWSER.exists()

Paid Features & Tier-Gated QA
  settings.html: 47147 bytes
  start.html:    28720 bytes
  schedule.html: 16783 bytes


In [2]:
# P1: Tier mentions -- settings.html shows Free/Starter/Pro with correct prices ($0/$8/$28)
print("=" * 60)
print("P1: Tier Mentions & Pricing")
print("=" * 60)

# settings.html must mention all three tiers
has_free_settings = "Free" in settings_html and "$0" in settings_html
record("P1.1", "settings.html mentions Free tier with $0",
       has_free_settings,
       detail="Free + $0 found" if has_free_settings else "Missing Free/$0 reference")

has_starter_settings = "Starter" in settings_html and "$8" in settings_html
record("P1.2", "settings.html mentions Starter tier with $8/mo",
       has_starter_settings,
       detail="Starter + $8 found" if has_starter_settings else "Missing Starter/$8 reference")

has_pro_settings = "Pro" in settings_html and "$28" in settings_html
record("P1.3", "settings.html mentions Pro tier with $28/mo",
       has_pro_settings,
       detail="Pro + $28 found" if has_pro_settings else "Missing Pro/$28 reference")

# Tier row in settings should show all three together
tier_row = re.search(r'Free.*Starter.*Pro|Tier.*Free.*Starter.*Pro', settings_html)
record("P1.4", "settings.html has combined tier display (Free/Starter/Pro in one row)",
       tier_row is not None,
       detail="Combined tier row found" if tier_row else "Tiers not shown together")

# start.html should also mention tiers in the offer notice
start_tiers = ("Free" in start_html and "Starter" in start_html and "Pro" in start_html)
record("P1.5", "start.html mentions all three tiers (Free/Starter/Pro)",
       start_tiers,
       detail="All tiers in start.html" if start_tiers else "Missing tier mentions in start page")

# start.html prices match
start_8 = "$8" in start_html
start_28 = "$28" in start_html
record("P1.6", "start.html has correct prices ($8/mo and $28/mo)",
       start_8 and start_28,
       detail=f"$8={start_8}, $28={start_28}")

p1_pass = sum(1 for r in results if r['id'].startswith('P1') and r['status'] == 'PASS')
p1_total = sum(1 for r in results if r['id'].startswith('P1'))
print(f"\nP1 complete: {p1_pass}/{p1_total} passed")

P1: Tier Mentions & Pricing
  PASS: settings.html mentions Free tier with $0
  PASS: settings.html mentions Starter tier with $8/mo
  PASS: settings.html mentions Pro tier with $28/mo
  PASS: settings.html has combined tier display (Free/Starter/Pro in one row)
  PASS: start.html mentions all three tiers (Free/Starter/Pro)
  PASS: start.html has correct prices ($8/mo and $28/mo)

P1 complete: 6/6 passed


In [3]:
# P2: Cloud twin references -- schedule or settings mentions cloud twin as Pro+ feature
print("=" * 60)
print("P2: Cloud Twin References")
print("=" * 60)

# Check schedule-cloud.js for Cloud Twin indicator
cloud_js = read_file(WEB / "js" / "schedule-cloud.js")
has_cloud_twin_js = "Cloud Twin" in cloud_js
record("P2.1", "schedule-cloud.js has Cloud Twin indicator text",
       has_cloud_twin_js,
       detail="Cloud Twin indicator with Connected/Disconnected/Not configured states" if has_cloud_twin_js else "No Cloud Twin text in schedule-cloud.js")

# Cloud twin states: Connected, Disconnected, Not configured
twin_states = ["Connected", "Disconnected", "Not configured"]
found_twin_states = [s for s in twin_states if f"Cloud Twin: {s}" in cloud_js]
record("P2.2", f"Cloud Twin has all 3 status states ({len(found_twin_states)}/3)",
       len(found_twin_states) == 3,
       detail=f"Found: {found_twin_states}")

# start.html mentions cloud twin as Pro feature
start_twin = re.search(r'Pro.*cloud twin|cloud twin.*Pro', start_html, re.IGNORECASE)
record("P2.3", "start.html associates cloud twin with Pro tier",
       start_twin is not None,
       detail="Pro + cloud twin linked in start page" if start_twin else "Cloud twin not linked to Pro in start page")

# schedule.html loads schedule-cloud.js
loads_cloud_js = "schedule-cloud.js" in schedule_html
record("P2.4", "schedule.html loads schedule-cloud.js",
       loads_cloud_js,
       detail="schedule-cloud.js script tag found" if loads_cloud_js else "schedule-cloud.js not loaded on schedule page")

# Cloud twin requires auth token (gated)
has_auth_gate = "isCloudEnabled" in cloud_js and "getAuthToken" in cloud_js
record("P2.5", "Cloud twin features gated behind auth token",
       has_auth_gate,
       detail="isCloudEnabled + getAuthToken checks present" if has_auth_gate else "No auth gating on cloud twin")

p2_pass = sum(1 for r in results if r['id'].startswith('P2') and r['status'] == 'PASS')
p2_total = sum(1 for r in results if r['id'].startswith('P2'))
print(f"\nP2 complete: {p2_pass}/{p2_total} passed")

P2: Cloud Twin References
  PASS: schedule-cloud.js has Cloud Twin indicator text
  PASS: Cloud Twin has all 3 status states (3/3)
  PASS: start.html associates cloud twin with Pro tier
  PASS: schedule.html loads schedule-cloud.js
  PASS: Cloud twin features gated behind auth token

P2 complete: 5/5 passed


In [4]:
# P3: BYOK vs Managed -- settings.html has API key input (BYOK), start.html mentions managed LLM
print("=" * 60)
print("P3: BYOK vs Managed LLM")
print("=" * 60)

# settings.html has BYOK section
has_byok = "BYOK" in settings_html
record("P3.1", "settings.html mentions BYOK (bring your own key)",
       has_byok,
       detail="BYOK reference found" if has_byok else "No BYOK mention in settings")

# settings.html has API key management
has_api_key_section = "API Key" in settings_html or "API key" in settings_html or "api_key" in settings_html
record("P3.2", "settings.html has API key management section",
       has_api_key_section,
       detail="API key section found" if has_api_key_section else "No API key section")

# settings.html mentions managed LLM provider (Together.ai)
has_managed = "Together.ai" in settings_html or "managed" in settings_html.lower()
record("P3.3", "settings.html mentions managed LLM provider",
       has_managed,
       detail="Together.ai / managed reference found" if has_managed else "No managed LLM mention")

# start.html mentions managed LLM as paid option
start_managed = "managed" in start_html.lower()
record("P3.4", "start.html mentions managed LLM option",
       start_managed,
       detail="Managed LLM reference in start page" if start_managed else "No managed LLM in start page")

# start.html mentions free with own API key
start_free_byok = re.search(r'[Ff]ree.*API key|API key.*free|own.*key', start_html)
record("P3.5", "start.html communicates free-with-own-key model",
       start_free_byok is not None,
       detail="Free + own key language found" if start_free_byok else "Free BYOK not communicated on start page")

# settings.html has rotate/revoke API key capability
has_rotate = "Rotate" in settings_html or "rotate" in settings_html
record("P3.6", "settings.html has API key rotation capability",
       has_rotate,
       detail="Rotate API key found" if has_rotate else "No key rotation option")

p3_pass = sum(1 for r in results if r['id'].startswith('P3') and r['status'] == 'PASS')
p3_total = sum(1 for r in results if r['id'].startswith('P3'))
print(f"\nP3 complete: {p3_pass}/{p3_total} passed")

P3: BYOK vs Managed LLM
  PASS: settings.html mentions BYOK (bring your own key)
  PASS: settings.html has API key management section
  PASS: settings.html mentions managed LLM provider
  PASS: start.html mentions managed LLM option
  PASS: start.html communicates free-with-own-key model
  PASS: settings.html has API key rotation capability

P3 complete: 6/6 passed


In [5]:
# P4: Evidence retention tiers -- some page mentions 90-day (Pro) evidence retention
print("=" * 60)
print("P4: Evidence Retention Tiers")
print("=" * 60)

# settings.html mentions 90-day retention
has_90_day = "90-day" in settings_html or "90 day" in settings_html
record("P4.1", "settings.html mentions 90-day evidence retention",
       has_90_day,
       detail="90-day retention found" if has_90_day else "No 90-day retention mention")

# settings.html has evidence/Part 11 section
has_evidence_section = "Part 11" in settings_html or "Evidence" in settings_html or "evidence" in settings_html
record("P4.2", "settings.html has evidence/compliance section",
       has_evidence_section,
       detail="Evidence/Part 11 section found" if has_evidence_section else "No evidence section")

# Cloud sync mentioned in settings
has_cloud_sync = "Cloud sync" in settings_html or "cloud sync" in settings_html
record("P4.3", "settings.html mentions cloud sync for evidence",
       has_cloud_sync,
       detail="Cloud sync reference found" if has_cloud_sync else "No cloud sync mention")

# Evidence Vault mentioned
has_vault = "Evidence Vault" in settings_html or "evidence vault" in settings_html.lower()
record("P4.4", "settings.html references Evidence Vault",
       has_vault,
       detail="Evidence Vault found" if has_vault else "No Evidence Vault reference")

# start.html mentions evidence as Pro feature
start_evidence = re.search(r'Pro.*evidence|evidence.*Pro', start_html, re.IGNORECASE)
record("P4.5", "start.html links evidence features to Pro tier",
       start_evidence is not None,
       detail="Pro + evidence linked on start page" if start_evidence else "Evidence not linked to Pro on start page")

# schedule.html has eSign tab (evidence management)
has_esign = 'data-view="esign"' in schedule_html or 'viewEsign' in schedule_html
record("P4.6", "schedule.html has eSign evidence tab",
       has_esign,
       detail="eSign tab found" if has_esign else "No eSign tab on schedule page")

p4_pass = sum(1 for r in results if r['id'].startswith('P4') and r['status'] == 'PASS')
p4_total = sum(1 for r in results if r['id'].startswith('P4'))
print(f"\nP4 complete: {p4_pass}/{p4_total} passed")

P4: Evidence Retention Tiers
  PASS: settings.html mentions 90-day evidence retention
  PASS: settings.html has evidence/compliance section
  PASS: settings.html mentions cloud sync for evidence
  PASS: settings.html references Evidence Vault
  PASS: start.html links evidence features to Pro tier
  PASS: schedule.html has eSign evidence tab

P4 complete: 6/6 passed


In [6]:
# P5: No tier-gated features accessible without auth indication
# Pages mentioning Pro features should also mention upgrade/login
print("=" * 60)
print("P5: Auth Gating -- Pro Features Require Login/Upgrade Path")
print("=" * 60)

# start.html has sign-in/auth buttons
has_auth_btns = ('id="btn-google"' in start_html or
                 'id="btn-github"' in start_html or
                 'id="btn-email"' in start_html)
record("P5.1", "start.html has authentication buttons (Google/GitHub/Email)",
       has_auth_btns,
       detail="Auth buttons found" if has_auth_btns else "No auth buttons on start page")

# start.html mentions 'free' alongside sign-in (no paywall before entry)
free_entry = re.search(r'free|no credit card', start_html, re.IGNORECASE)
record("P5.2", "start.html communicates free entry (no paywall)",
       free_entry is not None,
       detail="Free/no credit card language found" if free_entry else "No free entry messaging")

# settings.html with Pro features also has auth check in JS
settings_auth_check = re.search(r'solace_api_key|solace_auth_token|solace_token', settings_html)
record("P5.3", "settings.html checks for auth token (gating mechanism)",
       settings_auth_check is not None,
       detail="Auth token check found" if settings_auth_check else "No auth check in settings page")

# Custom theme editor is labeled as Pro
pro_theme = re.search(r'Pro.*users|Pro.*theme|custom.*theme.*Pro', settings_html, re.IGNORECASE)
record("P5.4", "Custom theme editor labeled as Pro feature",
       pro_theme is not None,
       detail="Pro label on custom theme found" if pro_theme else "Custom theme not labeled as Pro")

# schedule-core.js has tier-based eSign limits
core_js = read_file(WEB / "js" / "schedule-core.js")
has_tier_limits = "TIER_ESIGN_LIMITS" in core_js
record("P5.5", "schedule-core.js has tier-based eSign limits",
       has_tier_limits,
       detail="TIER_ESIGN_LIMITS found" if has_tier_limits else "No tier limits for eSign")

# Verify tier limits include free=0 and pro=Infinity
if has_tier_limits:
    tier_match = re.search(r'TIER_ESIGN_LIMITS.*?\}', core_js, re.DOTALL)
    tier_text = tier_match.group(0) if tier_match else ""
    free_zero = "free: 0" in tier_text or "free:0" in tier_text
    pro_inf = "pro: Infinity" in tier_text or "pro:Infinity" in tier_text
    record("P5.6", "eSign limits: free=0, pro=Infinity (correct gating)",
           free_zero and pro_inf,
           detail=f"free=0:{free_zero}, pro=Infinity:{pro_inf}")
else:
    record("P5.6", "eSign limits: free=0, pro=Infinity (correct gating)",
           False,
           detail="TIER_ESIGN_LIMITS not found, cannot verify")

# start.html has sign-out capability (revoke access)
has_signout = "signOut" in start_html or "sign out" in start_html.lower() or "Sign out" in start_html
record("P5.7", "start.html has sign-out / revoke access capability",
       has_signout,
       detail="Sign-out found" if has_signout else "No sign-out option")

p5_pass = sum(1 for r in results if r['id'].startswith('P5') and r['status'] == 'PASS')
p5_total = sum(1 for r in results if r['id'].startswith('P5'))
print(f"\nP5 complete: {p5_pass}/{p5_total} passed")

P5: Auth Gating -- Pro Features Require Login/Upgrade Path
  PASS: start.html has authentication buttons (Google/GitHub/Email)
  PASS: start.html communicates free entry (no paywall)
  PASS: settings.html checks for auth token (gating mechanism)
  PASS: Custom theme editor labeled as Pro feature
  PASS: schedule-core.js has tier-based eSign limits
  PASS: eSign limits: free=0, pro=Infinity (correct gating)
  PASS: start.html has sign-out / revoke access capability

P5 complete: 7/7 passed


In [7]:
# P6: Evidence Summary -- Paid Features QA
print("=" * 60)
print("P6: Evidence Summary -- NB33 Paid Features QA")
print("=" * 60)

total = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
failed = sum(1 for r in results if r["status"] == "FAIL")
score = (passed / total * 100) if total > 0 else 0

print(f"\nTotal probes: {total}")
print(f"PASS: {passed}")
print(f"FAIL: {failed}")
print(f"Score: {score:.1f}%")
print(f"Meets minimum ({MIN_SCORE}%): {'YES' if score >= MIN_SCORE else 'NO'}")

# Per-probe-group breakdown
probe_groups = {}
for r in results:
    group = r["id"].split(".")[0]
    if group not in probe_groups:
        probe_groups[group] = {"pass": 0, "fail": 0}
    if r["status"] == "PASS":
        probe_groups[group]["pass"] += 1
    else:
        probe_groups[group]["fail"] += 1

print("\n--- Per-Group Breakdown ---")
for group, counts in sorted(probe_groups.items()):
    g_total = counts["pass"] + counts["fail"]
    g_pct = counts["pass"] / g_total * 100 if g_total else 0
    status = "PASS" if counts["fail"] == 0 else "FAIL"
    print(f"  {group}: {counts['pass']}/{g_total} ({g_pct:.0f}%) [{status}]")

if failed > 0:
    print("\n--- Failures ---")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  {r['id']}: {r['name']} -- {r['detail']}")

# Evidence hash
evidence_blob = json.dumps(results, sort_keys=True)
evidence_hash = hashlib.sha256(evidence_blob.encode()).hexdigest()[:16]
print(f"\nEvidence hash: {evidence_hash}")
print(f"Timestamp: {datetime.now().isoformat()}")
print(f"Notebook: {NOTEBOOK_PATH}")
print(f"NORTHSTAR: {NORTHSTAR}")

assert score >= MIN_SCORE, f"Score {score:.1f}% below minimum {MIN_SCORE}%"
print(f"\n{'=' * 60}")
print(f"VERDICT: PASS ({score:.1f}% >= {MIN_SCORE}%)")
print(f"{'=' * 60}")

P6: Evidence Summary -- NB33 Paid Features QA

Total probes: 30
PASS: 30
FAIL: 0
Score: 100.0%
Meets minimum (70%): YES

--- Per-Group Breakdown ---
  P1: 6/6 (100%) [PASS]
  P2: 5/5 (100%) [PASS]
  P3: 6/6 (100%) [PASS]
  P4: 6/6 (100%) [PASS]
  P5: 7/7 (100%) [PASS]

Evidence hash: a93c47902380932a
Timestamp: 2026-03-06T10:27:23.771021
Notebook: notebooks/qa/33-paid-features-qa.ipynb
NORTHSTAR: paid-features-qa

VERDICT: PASS (100.0% >= 70%)
